# StreamForge cohorts, retention, and revenue

You continue as a junior data analyst at **StreamForge**. Leadership wants a clear picture of how new users behave over time and how engagement connects to revenue quality.

<img src ="https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Exploratory_Data_Analysis/M3_D2_Streamforge.png"/>

They ask you to **build time-aware cohort views** that mix user attributes, watch behavior, and payments. They want to read retention, ARPU, ARPPU, and how the engagement mix changes as users age on the platform.

You will use Python, NumPy, and Pandas to reshape events into user-month panels and then join revenue. You will work with three sources:

* `user_profiles.csv` with static attributes and monthly list price
* `viewing_events.csv` with watch behavior and timing
* `payments.csv` with payment dates, amounts, and statuses

## Your mission

Your goal is to construct a **cohort retention matrix** and link it to monthly revenue. You will define cohorts by signup month, label user-month activity and calculate retention.

<img src =""/>


## Task 1: Create cohorts and the user-month spine

1. Derive `signup_month` for each user. If you do not have explicit signup dates, you use the first observed payment date as a proxy. 

2. Build a user-month index from each user’s `signup_month` through the last month present in `payments`.

3. Store months as `Period("M")` to make arithmetic and joins safer.

4. Label each row with `months_since_signup` as an integer starting at zero.

**You now have a complete user-month spine that you can enrich with behavior and revenue.**


In [24]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load datasets
user_profiles = pd.read_csv("user_profiles.csv")
viewing_events = pd.read_csv("viewing_events.csv")
payments = pd.read_csv("payments.csv")

In [25]:
user_profiles.head() 

,user_id,subscription_type,user_age,country,monthly_revenue
0,user_001,premium,32,DE,13.99
1,user_002,standard,68,CA,8.99
2,user_003,premium,35,UK,17.99
3,user_004,premium,56,JP,8.99
4,user_005,premium,21,DE,13.99


In [26]:
viewing_events.head()

,user_id,timestamp,event_type,duration_seconds,watched_duration,total_duration,content_id,episode_number,watched_seconds,total_seconds,user_rating
0,user_041,2024-02-05 07:14:00,play,1443,1186,1862,content_015,1,1186,1862,3.7
1,user_035,2024-02-24 01:01:00,error,251,80,1203,content_012,1,80,1203,3.0
2,user_002,2024-03-24 22:34:00,pause,919,316,2758,content_072,1,316,2758,3.4
3,user_001,2024-03-30 13:21:00,pause,629,141,1510,content_098,1,141,1510,4.8
4,user_022,2024-02-18 03:22:00,play,3117,2018,6422,content_014,1,2018,6422,1.2


In [35]:
payments[payments["user_id"]=="user_001"]

,user_id,payment_date,amount,payment_status
54,user_001,2024-03-18,17.99,success
82,user_001,2024-12-07,8.99,success
96,user_001,2024-08-08,8.99,success
178,user_001,2024-06-09,17.99,pending
195,user_001,2024-08-20,8.99,success
202,user_001,2024-07-11,17.99,success
203,user_001,2024-01-10,8.99,failed


In [28]:
payments["payment_date"] = pd.to_datetime(payments["payment_date"])
viewing_events["timestamp"] = pd.to_datetime(viewing_events["timestamp"])
# convertir les colonnes de dates en datetime pour pouvoir extraire
# le mois, le jour, les périodes et les durées

In [29]:
# derive `signup_month` for each user from the first observed payment date as a proxy. 
# merge payments to user_profiles to get the signup month
# 1) Reduce payments to ONE row per user: earliest payment
# pour trouver le premier paiement, je filtre uniquement les paiements réussis
# je trie les paiements par date croissante, puis je regroupe par utilisateur
# ensuite j'approxime la première date de paiement à la date d'inscription avec .first()
# qui prend le premier paiement réussi
# on renomme la colonne payment_date en signup_month pour éviter les confusions dans le merge

first_pay = (
    payments[payments["payment_status"] == "success"]
    .sort_values("payment_date")
    .groupby("user_id", as_index=False)
    # .agg(signup_month=("signup_date", "first"))
    .first()[["user_id", "payment_date"]] # ["user_id"]["payment_date"]
    .rename(columns={"payment_date": "signup_date"})
)
print(first_pay.head())



    user_id signup_date
0  user_001  2024-03-18
1  user_002  2024-04-02
2  user_003  2024-01-02
3  user_004  2024-02-24
4  user_005  2024-03-07


In [30]:
# 2) Merge into profiles (now 1:1; no duplicates)
first_pay["signup_month"] = first_pay["signup_date"].dt.to_period("M")

profiles = user_profiles.merge(first_pay, on="user_id", how="left")
print(profiles.head())

    user_id subscription_type  user_age country  monthly_revenue signup_date  \
0  user_001           premium        32      DE            13.99  2024-03-18   
1  user_002          standard        68      CA             8.99  2024-04-02   
2  user_003           premium        35      UK            17.99  2024-01-02   
3  user_004           premium        56      JP             8.99  2024-02-24   
4  user_005           premium        21      DE            13.99  2024-03-07   

  signup_month  
0      2024-03  
1      2024-04  
2      2024-01  
3      2024-02  
4      2024-03  


In [31]:
# 3.
# find the last month in either payments or viewing_events
last_month_pay = payments["payment_date"].max().to_period("M")
last_month_view = viewing_events["timestamp"].max().to_period("M")
last_month = max(last_month_pay, last_month_view)



In [33]:

# 4. 
# Create a DataFrame to hold user-month combinations

# Store months as `Period("M")` to make arithmetic and joins safer.

# Label each row with `months_since_signup` as an integer starting at zero.

# You now have a complete user-month spine that you can enrich with behavior and revenue.
rows = []

# point de départ : j'ai besoin de stocker des lignes et des colonnes 
# dans un dataframe => je crée une liste vide des lignes

for _, row in profiles.iterrows():
    # mes données sont dans profiles, j'itère sur chaque ligne avec iterrows()
    # _ signifie 
    uid = row["user_id"]
    sm = row["signup_month"]

    if pd.isna(sm):
        continue # je saute les utilisateurs sans date d'inscription

    # all months from signup_month → last_month
    months = pd.period_range(sm, last_month, freq="M")
    # c'est ici que je génère une liste de sm à last_month avec une fréquence mensuelle
    # exemple months = [2024-03, 2024-04, 2024-05, … , 2024-12]

    for i, m in enumerate(months):
        rows.append([uid, m, i]) 
        # pour chaque utilisateur
        # l'index i dans la liste months représente automatiquement la durée "months_since_signup"
        # m représente 2024-03 par exemple

user_month_spine = pd.DataFrame(rows, columns=["user_id", "month", "months_since_signup"])
# enfin transformation de rows en dataframe avec des colonnes nommées

# user_month_spine.head()
print(user_month_spine[user_month_spine["user_id"]=="user_001"])

    user_id    month  months_since_signup
0  user_001  2024-03                    0
1  user_001  2024-04                    1
2  user_001  2024-05                    2
3  user_001  2024-06                    3
4  user_001  2024-07                    4
5  user_001  2024-08                    5
6  user_001  2024-09                    6
7  user_001  2024-10                    7
8  user_001  2024-11                    8
9  user_001  2024-12                    9


## Task 2: Define activity and compute retention

1. Join viewing activity to the user-month spine by user and month.

2. Mark `is_active` as true when a user has at least one viewing session in that month or when the user has a successful payment that month.

3. Compute a cohort retention matrix with one row per `signup_month` and one column per `months_since_signup`. Each cell shows the percentage of the original cohort that remains active.


In [13]:
# Join viewing activity to the user-month spine by user and month.
# Aggregate viewing events → number of events per user and per month
viewing_events["month"] = viewing_events["timestamp"].dt.to_period("M")

view_agg = (
    viewing_events.groupby(["user_id", "month"])
    .size() # size() compte le nombre d'évènements par groupe
    .reset_index(name="view_count")
)

# LEFT JOIN spine + viewing counts
spine = user_month_spine.merge(view_agg, on=["user_id", "month"], how="left")
spine["view_count"] = spine["view_count"].fillna(0)
print(spine.head())


    user_id    month  months_since_signup  view_count
0  user_001  2024-03                    0         4.0
1  user_001  2024-04                    1         0.0
2  user_001  2024-05                    2         0.0
3  user_001  2024-06                    3         0.0
4  user_001  2024-07                    4         0.0


In [14]:
payments["month"] = payments["payment_date"].dt.to_period("M")

pay_success = payments[payments["payment_status"] == "success"]

pay_agg = (
    pay_success.groupby(["user_id", "month"])
    .size()
    .reset_index(name="successful_payments")
)

spine = spine.merge(pay_agg, on=["user_id", "month"], how="left")
spine["successful_payments"] = spine["successful_payments"].fillna(0)
print(spine.head())

    user_id    month  months_since_signup  view_count  successful_payments
0  user_001  2024-03                    0         4.0                  1.0
1  user_001  2024-04                    1         0.0                  0.0
2  user_001  2024-05                    2         0.0                  0.0
3  user_001  2024-06                    3         0.0                  0.0
4  user_001  2024-07                    4         0.0                  1.0


In [ ]:
# Mark `is_active` as true when a user has at least one viewing session in that month or when the user has a successful payment that month.
spine["is_active"] = (spine["view_count"] > 0) | (spine["successful_payments"] > 0)
print(spine.head())


,user_id,month,months_since_signup,is_active
0,user_001,2024-01,0,True
1,user_001,2024-02,1,True
2,user_001,2024-03,2,True
3,user_001,2024-04,3,False
4,user_001,2024-05,4,False
...,...,...,...,...
515,user_050,2024-08,3,False
516,user_050,2024-09,4,False
517,user_050,2024-10,5,False
518,user_050,2024-11,6,False


In [ ]:
# Compute a cohort retention matrix with one row per `signup_month` and one column per `months_since_signup`. Each cell shows the percentage of the original cohort that remains active.
# 0) Get each user's signup_month from the spine
spine = spine.merge(
    profiles[["user_id", "signup_month"]],
    on="user_id",
    how="left"
)


# 1) Cohort sizes: unique users per signup_month
cohort_sizes = (
    profiles.groupby("signup_month")["user_id"]
    .nunique()
    .rename("cohort_size")
)


# 2) Active users per (signup_month, months_since_signup)

active_by_cohort = (
    spine[spine["is_active"]]
    .groupby(["signup_month", "months_since_signup"])["user_id"]
    .nunique()
    .rename("active_users")
    .reset_index()
)

# 3) Retention matrix (% of original cohort)



months_since_signup,0,1,2,3,4,5,6,7,8,9,10,11
signup_month,,,,,,,,,,,,
2024-01,100.0,100.0,100.0,57.1,42.9,35.7,21.4,42.9,28.6,64.3,28.6,28.6
2024-02,100.0,100.0,55.6,38.9,50.0,50.0,33.3,16.7,33.3,38.9,38.9,0.0
2024-03,100.0,25.0,25.0,37.5,25.0,50.0,37.5,50.0,25.0,12.5,0.0,0.0
2024-04,100.0,100.0,33.3,33.3,0.0,66.7,33.3,66.7,33.3,0.0,0.0,0.0
2024-05,100.0,66.7,66.7,33.3,33.3,33.3,0.0,33.3,0.0,0.0,0.0,0.0
2024-06,100.0,100.0,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-07,100.0,100.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024-08,100.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Task 3: Save artifacts 

Export 2 CSVs with no index: `user_month_spine.csv` and `retention_matrix.csv`.


In [ ]:
# export four CSVs with no index: `user_month_spine.csv` and `retention_matrix.csv`

